# Kelompok 06 · Training di Google Colab

Sebelum run:
1. Upload folder `train/` (5 folder kelas) ke Google Drive: `MyDrive/train/`.
2. **Runtime → Change runtime type → T4 GPU**.

`model.pth` dan `experiments/` otomatis disalin ke `MyDrive/hasil/`.

In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive'
!ls -lh {DRIVE_DIR}/train

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
lrw------- 1 root root 0 Oct 12 10:52 /content/drive/MyDrive/train -> /content/drive/.shortcut-targets-by-id/1_GMs_uJUpCAsCL-II-vCmVR0MEqosJIk/train


In [ ]:
# copy dataset dari Drive ke disk Colab
!mkdir -p /content/data
!cp -r {DRIVE_DIR}/train /content/data/
DATA = '/content/data/train'

import os
for c in sorted(os.listdir(DATA)):
    print(c, len(os.listdir(os.path.join(DATA, c))))

botol_plastik 523
gelas_plastik 521
kertas_kardus 365
saset_kemasan 673
styrofoam 671


In [ ]:
!git clone https://github.com/beningapniprameswari/Studio-2_Kelompok-6.git /content/Studio-2_Kelompok-6
%cd /content/Studio-2_Kelompok-6
!pip install -q -r submission/requirements.txt
!mkdir -p experiments
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/Studio-2_Kelompok-6'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 97 (delta 8), reused 5 (delta 4), pack-reused 82 (from 3)
Receiving objects: 100% (97/97), 147.55 MiB | 24.55 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Studio-2_Kelompok-6
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Baseline ResNet-18 (opsional)
Bobot masuk ke `baseline/model.pth`

In [ ]:
!python train_baseline.py --data {DATA} --epochs 8 2>&1 | tee experiments/01_baseline_resnet18.txt
!python evaluate.py --submission ./baseline --data ./my_val 2>&1 | tee -a experiments/01_baseline_resnet18.txt

kelas: ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam'] | train: 2340 | val: 413
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]
epoch  1  val_acc=0.8305
epoch  2  val_acc=0.8668
epoch  3  val_acc=0.8814
epoch  4  val_acc=0.9249
epoch  5  val_acc=0.9395
epoch  6  val_acc=0.9298
epoch  7  val_acc=0.9298
epoch  8  val_acc=0.9298
best val_acc=0.9395 -> baseline/model.pth
device=cuda  gambar=413

Macro-F1 : 0.9389
Akurasi  : 0.9395
  F1 botol_plastik   0.9740
  F1 gelas_plastik   0.9441
  F1 kertas_kardus   0.9159
  F1 saset_kemasan   0.9137
  F1 styrofoam       0.9469
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [75, 0, 0, 2, 1]
  gelas_plastik   [0, 76, 1, 0, 1]
  kertas_kardus   [0, 2, 49, 3, 1]
  saset_kemasan   [1, 4, 1, 90, 5]
  styrofoam       [0, 1, 1, 1, 98]


## 2. ConvNeXt-Tiny
Train di 85% data, epoch terbaik dipilih dari Macro-F1 di `my_val/`.

In [ ]:
!python train.py --data {DATA} --epochs 10 --workers 2 2>&1 | tee experiments/03_convnext_tiny_10ep_gpu.txt

device=cuda | train=2340 | val=413
epoch  1  loss=1.0665  (30s)  val_macroF1=0.9038  val_acc=0.9080  *
epoch  2  loss=0.8002  (23s)  val_macroF1=0.9381  val_acc=0.9419  *
epoch  3  loss=0.7561  (22s)  val_macroF1=0.9592  val_acc=0.9613  *
epoch  4  loss=0.6414  (22s)  val_macroF1=0.9516  val_acc=0.9540
epoch  5  loss=0.6419  (22s)  val_macroF1=0.9688  val_acc=0.9709  *
epoch  6  loss=0.6640  (22s)  val_macroF1=0.9611  val_acc=0.9637
epoch  7  loss=0.6671  (22s)  val_macroF1=0.9616  val_acc=0.9637
epoch  8  loss=0.6102  (22s)  val_macroF1=0.9643  val_acc=0.9661
epoch  9  loss=0.6788  (22s)  val_macroF1=0.9665  val_acc=0.9685
epoch 10  loss=0.6021  (22s)  val_macroF1=0.9665  val_acc=0.9685

terbaik val Macro-F1=0.9688 -> submission/model.pth
  F1 botol_plastik   0.9872
  F1 gelas_plastik   0.9615
  F1 kertas_kardus   0.9444
  F1 saset_kemasan   0.9706
  F1 styrofoam       0.9802
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [77, 1, 0, 0, 0]
  gelas_plastik   

In [ ]:
# cek submission pakai evaluate.py, lalu simpan hasil ke Drive
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | tee -a experiments/03_convnext_tiny_10ep_gpu.txt
!ls -lh submission/model.pth
!mkdir -p {DRIVE_DIR}/hasil
!cp submission/model.pth {DRIVE_DIR}/hasil/model.pth
!cp -r experiments {DRIVE_DIR}/hasil/
!ls -lhR {DRIVE_DIR}/hasil

device=cuda  gambar=413

Macro-F1 : 0.9688
Akurasi  : 0.9709
  F1 botol_plastik   0.9872
  F1 gelas_plastik   0.9615
  F1 kertas_kardus   0.9444
  F1 saset_kemasan   0.9706
  F1 styrofoam       0.9802
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [77, 1, 0, 0, 0]
  gelas_plastik   [1, 75, 0, 1, 1]
  kertas_kardus   [0, 1, 51, 2, 1]
  saset_kemasan   [0, 0, 2, 99, 0]
  styrofoam       [0, 1, 0, 1, 99]
-rw-r--r-- 1 root root 54M Oct 21 10:53 submission/model.pth
/content/drive/MyDrive/hasil:
total 54M
drwx------ 2 root root 4.0K Oct 21 10:55 experiments
-rw------- 1 root root  54M Oct 21 10:55 model.pth

/content/drive/MyDrive/hasil/experiments:
total 8.0K
-rw------- 1 root root 1.2K Oct 21 10:55 01_baseline_resnet18.txt
-rw------- 1 root root 1.1K Oct 21 10:55 02_convnext_tiny.txt
-rw------- 1 root root 1.6K Oct 21 10:55 03_convnext_tiny_10ep_gpu.txt
-rw------- 1 root root 3.0K Oct 21 10:55 04_convnext_tiny_15ep_cpu.txt


## 3. Train di seluruh data (Preview 2 / Final)

In [ ]:
EXP = '05_convnext_tiny_10ep_full'
!python train.py --data {DATA} --epochs 10 --workers 2 --full 2>&1 | tee experiments/{EXP}.txt

device=cuda | train=2753 | val=0
/content/Studio-2_Kelompok-6/train.py:126: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()
epoch  1  loss=1.0751  (33s)
epoch  2  loss=0.7775  (31s)
epoch  3  loss=0.7392  (30s)
epoch  4  loss=0.6594  (30s)
epoch  5  loss=0.6793  (31s)
epoch  6  loss=0.6483  (31s)
epoch  7  loss=0.6496  (31s)
epoch  8  loss=0.6490  (30s)
epoch  9  loss=0.6022  (30s)
epoch 10  loss=0.5936  (31s)
model epoch terakhir disimpan -> submission/model.pth


In [ ]:
# skor ini tidak valid (my_val ikut dilatih), cuma cek model.pth bisa dimuat evaluate.py
!python -c "from train import make_local_split; make_local_split('{DATA}', '.', 0.15, 42)"
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | grep -E "Macro|Akurasi|GAGAL|Error"
!ls -lh submission/model.pth
!mkdir -p {DRIVE_DIR}/hasil_full
!cp submission/model.pth {DRIVE_DIR}/hasil_full/model.pth
!cp experiments/{EXP}.txt {DRIVE_DIR}/hasil_full/
!ls -lh {DRIVE_DIR}/hasil_full

Macro-F1 : 1.0000
Akurasi  : 1.0000
-rw-r--r-- 1 root root 54M Oct 21 11:23 submission/model.pth
total 54M
-rw------- 1 root root 963 Oct 21 11:30 05_convnext_tiny_10ep_full.txt
-rw------- 1 root root 54M Oct 21 11:30 model.pth
